# Task 1 Statistics Evidence - jzho0172


## Setup


In [1]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "jzho0172"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

,unikey,project_root,raw_task1_csv,processed_task1_cleaned_csv
0,jzho0172,/home/kscii/Codes/data2001-group-assignment,/home/kscii/Codes/data2001-group-assignment/da...,/home/kscii/Codes/data2001-group-assignment/da...


## Shared Cleaning Input


In [2]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

,measure_code,parent_description,description,unit,2011_is_outlier,2015_is_outlier,2016_is_outlier,2017_is_outlier,2018_is_outlier,2019_is_outlier,2020_is_outlier,2021_is_outlier,2022_is_outlier,2023_is_outlier,2024_is_outlier,2025_is_outlier,outlier_count,has_outlier,year,value
0,CENSUS_34,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples,no.,True,False,False,False,False,False,False,False,False,False,False,False,1,True,2011,172620.0
1,CENSUS_2,Aboriginal and Torres Strait Islander Peoples ...,Aboriginal and Torres Strait Islander Peoples,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,2.5
2,CENSUS_15,Religious affiliation - Census,Buddhism,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,2.9
3,CENSUS_16,Religious affiliation - Census,Christianity,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,64.5
4,CENSUS_17,Religious affiliation - Census,Hinduism,%,False,False,False,False,False,False,False,False,False,False,False,False,0,False,2011,1.7


,rows,columns
0,2874,20


## Individual Derived Statistics


In [3]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"})
        continue
    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors))

,statistic_id,title,value,unit,description
0,jzho0172-1,"Estimated resident population growth, 2019-2024",5.38,percent,"Estimated resident population increased from 8,046,748 in 2019 to 8,479,314 in 2024, representin..."
1,jzho0172-2,"Population density change, 2019-2024",0.60,persons/km2,"Population density changed from 10.0 persons/km2 in 2019 to 10.6 persons/km2 in 2024, a change o..."
2,jzho0172-3,"Female share of estimated resident population, 2024",50.26,percent,"In 2024, females accounted for 50.26% of the estimated resident population (4,261,453 out of 8,4..."
3,jzho0172-4,"Female-male population gap, 2024",43592.00,people,"In 2024, the female population was 4,261,453, compared with 4,217,861 males. The female-male pop..."
4,jzho0172-5,"Median male age change, 2019-2024",0.70,years,"The median age of males changed from 36.8 years in 2019 to 37.5 years in 2024, a change of 0.70 ..."


## Key Findings

My derived statistics mainly focus on population growth and demographic structure in NSW.

The estimated resident population increased by 5.38% from 2019 to 2024, showing that the population continued to grow over the available period. Population density also increased slightly by 0.60 persons/km², which suggests a small rise in population concentration.

The 2024 population was relatively balanced by gender, with females making up 50.26% of the estimated resident population. However, the female population was still higher than the male population by 43,592 people.

The median age of males increased by 0.70 years from 2019 to 2024, suggesting a slight ageing trend in the male population. Overall, these findings provide useful background for understanding future demand for services and resources in NSW.

## Explanation Notes
These five derived statistics were calculated from the cleaned NSW dataset using Pandas. I selected population and demographic indicators because they provide useful context for later POI and well-resourced score analysis.

Population growth and population density help show whether service demand may be increasing. A growing population may require more access to facilities such as schools, health services, transport, and community infrastructure.

The female share and female-male population gap describe the demographic structure of the population. These indicators help show whether the population is relatively balanced by gender.

The median male age change provides a simple indicator of demographic ageing. Even a small increase in median age may be relevant when interpreting demand for age-related services.

Together, these statistics support the broader group analysis by giving background information about population size, density, and demographic structure before comparing POI distribution and resource scores.